# Project 1 · Phase 3 — Leakage review, uncertainty & Power BI exports

**Scope:** Historical, provider-level Kaggle benchmark. This notebook evaluates the existing 21 engineered features, tests a train/validation/test workflow, reports bootstrap uncertainty, and exports reproducible aggregate data for Power BI. **It does not validate a live prepayment system.**

**Important:** The split is random at provider level, not time-forward. Features aggregate historical claims, and the fraud labels are provider-level. Claims chronology, label provenance and prospective feature availability remain unresolved; do not describe these results as prospective performance.


In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve
from sklearn.inspection import permutation_importance
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
df=pd.read_csv(OUT/'provider_level_features.csv')
features=[c for c in df if c not in ['Provider','PotentialFraud','Label']]
assert df.Provider.is_unique and set(df.Label.unique())=={0,1}
print('Providers:',len(df),'features:',len(features),'positive labels:',int(df.Label.sum()))
print('Missing values:',int(df[features].isna().sum().sum()))
print('Duplicate provider identifiers:',int(df.Provider.duplicated().sum()))
print('Features:',features)


Providers: 5410 features: 21 positive labels: 506
Missing values: 0
Duplicate provider identifiers: 0
Features: ['NumClaims', 'NumUniqueBeneficiaries', 'TotalReimbursed', 'AvgReimbursedPerClaim', 'StdReimbursedPerClaim', 'MaxReimbursedClaim', 'TotalDeductible', 'AvgClaimDurationDays', 'AvgLengthOfStayDays', 'AvgNumDiagCodes', 'AvgNumProcCodes', 'PhysicianRoleOverlapRate', 'NumUniqueAttendingPhysicians', 'NumUniqueOperatingPhysicians', 'PctInpatient', 'AvgBeneficiaryAge', 'AvgChronicConditions', 'PctDeceasedBeneficiaries', 'PctRenalDisease', 'NumUniqueStates', 'ClaimsPerBeneficiary']


## Leakage-risk checklist

- **Provider overlap:** explicitly excluded between train, validation and test.
- **Target leakage:** `Provider`, `PotentialFraud`, and `Label` excluded from model features.
- **Temporal leakage:** **not ruled out**. Aggregates are based on full historical claim windows; no as-of prediction date has been defined.
- **Label construction:** Kaggle `PotentialFraud` labels require source/provenance review; they are not independently adjudicated claims in this audit.
- **Generalizability:** a random provider holdout is not an external payer or future-period validation.
- **Potential proxies:** reimbursement totals, claim volume, and patient characteristics may reflect case mix; investigate subgroup bias before deployment.


In [2]:
X=df[features]; y=df.Label
idx_dev,idx_test=train_test_split(np.arange(len(df)),test_size=.20,random_state=42,stratify=y)
idx_train,idx_val=train_test_split(idx_dev,test_size=.25,random_state=43,stratify=y.iloc[idx_dev])
assert not(set(idx_train)&set(idx_val) or set(idx_train)&set(idx_test) or set(idx_val)&set(idx_test))
models={
 'Logistic regression':make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=1500,random_state=42)),
 'Gradient boosting':make_pipeline(SimpleImputer(strategy='median'),GradientBoostingClassifier(random_state=42))}
for model in models.values():model.fit(X.iloc[idx_train],y.iloc[idx_train])

def select_threshold(y_val, scores, target=.70):
    fpr,tpr,thresholds=roc_curve(y_val,scores)
    good=np.flatnonzero((tpr>=target)&np.isfinite(thresholds))
    return float(thresholds[good[np.argmin(fpr[good])]])

rows=[]; prediction_rows=[]; model_scores={}
for name,model in models.items():
    val_scores=model.predict_proba(X.iloc[idx_val])[:,1]
    test_scores=model.predict_proba(X.iloc[idx_test])[:,1]
    threshold=select_threshold(y.iloc[idx_val],val_scores)
    predictions=(test_scores>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y.iloc[idx_test],predictions,labels=[0,1]).ravel()
    rows.append(dict(Model=name,ThresholdFromValidation=threshold,TestProviders=len(idx_test),Positives=int(y.iloc[idx_test].sum()),ROC_AUC=roc_auc_score(y.iloc[idx_test],test_scores),PR_AUC=average_precision_score(y.iloc[idx_test],test_scores),Precision=tp/max(tp+fp,1),Recall=tp/max(tp+fn,1),FalsePositiveRate=fp/max(fp+tn,1),TruePositive=tp,FalsePositive=fp,TrueNegative=tn,FalseNegative=fn))
    model_scores[name]=test_scores
    for i,score,pred in zip(idx_test,test_scores,predictions):prediction_rows.append(dict(Provider=df.Provider.iloc[i],Model=name,RiskScore=float(score),PredictedFlag=int(pred),ActualProviderLabel=int(y.iloc[i])))
metrics=pd.DataFrame(rows); print(metrics.round(4).to_string(index=False))
metrics.to_csv(OUT/'powerbi_phase3_model_metrics.csv',index=False)
pred_df=pd.DataFrame(prediction_rows); pred_df.to_csv(OUT/'powerbi_phase3_provider_scores.csv',index=False)


              Model  ThresholdFromValidation  TestProviders  Positives  ROC_AUC  PR_AUC  Precision  Recall  FalsePositiveRate  TruePositive  FalsePositive  TrueNegative  FalseNegative
Logistic regression                   0.6595           1082        101   0.9593  0.7489     0.5000  0.8119             0.0836            82             82           899             19
  Gradient boosting                   0.2248           1082        101   0.9570  0.6922     0.5532  0.7723             0.0642            78             63           918             23


## Bootstrap uncertainty

The following percentile intervals are **exploratory** bootstrap estimates over the fixed held-out test providers. Model training and validation thresholds are held fixed. They do not account for uncertainty from dataset sampling, label quality, model selection, or external distribution shift.


In [3]:
rng=np.random.default_rng(20261118)
truth=y.iloc[idx_test].to_numpy(); n=len(truth); B=400
ci_rows=[]
for name,scores in model_scores.items():
    for metric,func in [('ROC_AUC',roc_auc_score),('PR_AUC',average_precision_score)]:
        draws=[]
        for _ in range(B):
            sampled=rng.integers(0,n,n)
            if len(np.unique(truth[sampled]))<2:continue
            draws.append(func(truth[sampled],scores[sampled]))
        ci_rows.append(dict(Model=name,Metric=metric,Estimate=func(truth,scores),CI_Lower=np.quantile(draws,.025),CI_Upper=np.quantile(draws,.975),BootstrapReplicates=len(draws)))
ci=pd.DataFrame(ci_rows);print(ci.round(4).to_string(index=False));ci.to_csv(OUT/'powerbi_phase3_bootstrap_intervals.csv',index=False)


              Model  Metric  Estimate  CI_Lower  CI_Upper  BootstrapReplicates
Logistic regression ROC_AUC    0.9593    0.9453    0.9713                  400
Logistic regression  PR_AUC    0.7489    0.6717    0.8198                  400
  Gradient boosting ROC_AUC    0.9570    0.9444    0.9706                  400
  Gradient boosting  PR_AUC    0.6922    0.5904    0.7975                  400


## Explainability and risk-band summaries

Permutation importance is evaluated on the validation split using PR-AUC, not on the test split. It measures model dependence on each feature in this dataset; it is **not causal attribution**. Risk scores are model outputs and should not be labeled calibrated probabilities without calibration testing.


In [4]:
importance_rows=[]
for name,model in models.items():
    perm=permutation_importance(model,X.iloc[idx_val],y.iloc[idx_val],scoring='average_precision',n_repeats=4,random_state=42,n_jobs=1)
    for f,m,s in zip(features,perm.importances_mean,perm.importances_std):
        importance_rows.append(dict(Model=name,Feature=f,MeanPRAUCDrop=m,StdPRAUCDrop=s))
imp=pd.DataFrame(importance_rows).sort_values(['Model','MeanPRAUCDrop'],ascending=[True,False]);imp.to_csv(OUT/'powerbi_phase3_feature_importance.csv',index=False)
print(imp.groupby('Model').head(8).round(4).to_string(index=False))

pred_df['RiskBand']=pd.cut(pred_df.RiskScore,[-.00001,.2,.4,.6,.8,1.00001],labels=['0–20','20–40','40–60','60–80','80–100'])
summary=pred_df.groupby(['Model','RiskBand','ActualProviderLabel'],observed=True).agg(Providers=('Provider','size'),Flagged=('PredictedFlag','sum')).reset_index()
summary.to_csv(OUT/'powerbi_phase3_risk_bands.csv',index=False)
print('Saved 5 Power BI CSV exports.')


              Model                      Feature  MeanPRAUCDrop  StdPRAUCDrop
  Gradient boosting              TotalReimbursed         0.2402        0.0380
  Gradient boosting              TotalDeductible         0.2312        0.0071
  Gradient boosting         ClaimsPerBeneficiary         0.0586        0.0136
  Gradient boosting                    NumClaims         0.0401        0.0061
  Gradient boosting              AvgNumDiagCodes         0.0244        0.0139
  Gradient boosting          AvgLengthOfStayDays         0.0159        0.0119
  Gradient boosting NumUniqueOperatingPhysicians         0.0132        0.0099
  Gradient boosting                 PctInpatient         0.0124        0.0028
Logistic regression                    NumClaims         0.2411        0.0179
Logistic regression              TotalReimbursed         0.1137        0.0105
Logistic regression        StdReimbursedPerClaim         0.0448        0.0099
Logistic regression              AvgNumDiagCodes         0.0325 

## Interpretation and limitations

1. Compare the two models at their **validation-selected thresholds**. Their actual test recall may differ; a lower false-positive rate alone is **not** a matched-recall improvement.
2. The provider fraud label is not a claim-level fraud finding. Do not describe risk-band counts as confirmed fraudulent claims.
3. Do not release beneficiary-level raw claims in public dashboards. This notebook exports provider-level synthetic/benchmark identifiers and aggregated evaluation data; verify dataset licensing and disclosure requirements before publishing.
4. The next research step is a time-aware as-of feature pipeline, label-provenance review, subgroup analysis, and external validation if lawful data access becomes available.
5. Record analysis dates and methodology changes.
